# Optimizing Agentic AI Collaboration v3.3.1 — Phi-4-mini-instruct Self-Consistency
## MMLU only

**Model:** `microsoft/Phi-4-mini-instruct` with 4-bit NF4 quantization via bitsandbytes.
**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier).

## Project aims

 1. a LangChain/LangGraph-based multi-agent Society-of-Minds prototype supporting ≥3 interaction protocols,
 2. benchmark gains (≥5% on MATH-500 and MMLU vs a single-agent SLM baseline), and
 3. a cost/benefit profile using latency + total token consumption, all tracked in W&B.

## What this notebook is

This notebook runs the **Self-Consistency (SC)** protocol for **MMLU** using **Phi-4-mini-instruct** — the *only fresh generator compute* in the L1 campaign. It draws the same fixed 128-question subset as the `v2.3.0` single-agent baseline (`data_seed=789`) and generates **5 independent samples per question**, one per seed (`42, 123, 456, 654, 987`). From that single pool it reports the full **SC@k ladder — SC@1, SC@3, SC@4, SC@5** — as the mean majority-vote accuracy over **every** size-k subset of the 5 generations (`C(5,1)=5`, `C(5,3)=10`, `C(5,4)=5`, `C(5,5)=1`), at zero extra generation cost.

Every full generation (not just a truncated display column) is persisted as a W&B artifact — this is what lets **Majority Voting** and **BoN-MAV** reuse these exact responses later without regenerating anything.

## Relation with the project WPs

**WP2 - T4: L1 multi-agent protocol — Self-Consistency.** This notebook is the sole generator-compute stage of the L1 campaign; Majority Voting and BoN-MAV consume its persisted generations directly. Its latency/token/VRAM measurements, composed per SC@k subset, feed the WP3 - T1 compute-accuracy trade-off analysis.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to one GPU).
2. Add the following secrets via **Add-ons → Secrets**, enabling each for the notebook:
   - `GITHUB_TOKEN` — GitHub PAT with repo read scope
   - `HF_TOKEN` — Hugging Face read-access token
   - `WANDB_API_KEY` — Weights & Biases API key
3. Click **Run All**. The first cell auto-detects Kaggle, clones the repo, and loads secrets.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets

## What this notebook implements

**Self-Consistency** over **MMLU** with full W&B tracking:

* **5 independent generations per question** (one per seed), sampled at `top_k=40` (Wang et al., *Self-Consistency*) rather than the baselines' `top_k=20` — a deliberate, documented deviation, so SC generations are *not* byte-identical to the `v2.3.0` baseline's.
* Generation and voting are both orchestrated through the **same LangGraph template** (`utils.graphs.build_majority_voting_graph`): a single live agent for generation, and one compiled replay graph per subset of the SC ladder for aggregation. The majority vote is produced **by the graph's `aggregate` node itself** (via `utils.voting.group_generations` + `majority_vote`), not by a Python loop that merely reuses its data — so every reported SC number is a number the protocol actually computed.
* Every full response (not truncated) is persisted as a `sc-generations-*` **W&B artifact** — the reuse vehicle for Majority Voting and BoN-MAV.
* MMLU's exact-letter grouping and MATH-500's LaTeX-equivalence grouping are two configurations of one unified path — `utils.voting.MMLU_SPEC` (`SPEC` in this notebook) binds the benchmark's extraction/equivalence/scoring rules once (no `math_verify` involved), and `utils.voting.run_self_consistency` drives the full **SC@1 / SC@3 / SC@4 / SC@5 ladder** from it. Each subset's own replay graph groups only that subset's members.
* Ties (a subset's top-voted letters all equally represented) are broken by a **uniform random choice** drawn from one seeded `random.Random(TIE_BREAK_SEED)` shared across every vote in the ladder — reproducible end-to-end from a fresh seed.

No overhead ratios are computed here — those are SC-vs-baseline / MAJ-vs-baseline comparisons for the offline analysis stage.

## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `langgraph`, `datasets`, `wandb`, `math-verify`). If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [1]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

Cloning into '/kaggle/working/tfg-agentic-ai'...


✅ Repo cloned to /kaggle/working/tfg-agentic-ai
repo_root = /kaggle/working/tfg-agentic-ai


In [2]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

/kaggle/working/tfg-agentic-ai
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: kaggle
✅ Secrets loaded from Kaggle Secrets.


## 3. Define Run Metadata

`GEN` matches the `v2.3.0` baseline's sampling parameters (`temperature=0.7`, `top_p=0.95`, `max_new_tokens=1024`) except `top_k`, deliberately raised to `40` per Wang et al.'s Self-Consistency paper. `CONFIG` carries the same `eval_size`/`data_seed` as the baseline (so the fixed 128-question subset is identical), plus the SC-specific `SC_SEEDS` (5 generation seeds), `TIE_BREAK_SEED`, and `K_LADDER`.

In [4]:
from utils.registry import MODELS

MODEL_KEY = "phi-4-mini"           # the ONE model literal in this notebook
DATASET   = "mmlu"

MODEL    = MODELS[MODEL_KEY]
RUN_NAME = MODEL["run_name"]

PROJECT = "agentic-final-results" if (ENVIRONMENT == "kaggle") else "dev-agentic-baselines"
print(f"W&B Project: {PROJECT}")

# Same sampling params as the v2.3.0 baseline (T=0.7, top_p=0.95, max_new_tokens=1024),
# except top_k=40 (Wang et al., Self-Consistency) instead of the baseline's 20 — SC
# generations are therefore NOT byte-identical to v2.3.0's; this is a deliberate,
# documented deviation, recorded here and in every persisted manifest.
GEN = {
    "max_new_tokens": 1024,
    "temperature":    0.7,
    "top_p":          0.95,
    "top_k":          40,
}

# 5 distinct generation seeds -> the single pool the whole SC@k ladder is read off
# of. gen_id (0..4) is this list's index everywhere: artifacts, tables, tie-break keys.
# Kept as a literal here rather than centralised in utils/ — every notebook pins its
# own copy so the pool is visible at the point of use; it still reaches W&B lineage
# via CONFIG["generation_seeds"] below.
SC_SEEDS       = [42, 123, 456, 654, 987]
TIE_BREAK_SEED = 2026
K_LADDER       = [1, 3, 4, 5]

CONFIG = {
    "model_key":        MODEL_KEY,
    "model_id":         MODEL["model_id"],
    "quantization":     "nf4-4bit",
    "framework":        "langchain+langgraph",
    "protocol":         "self-consistency",
    "eval_size":        128,
    "data_seed":        789,
    "generation_seeds": SC_SEEDS,
    "tie_break_seed":   TIE_BREAK_SEED,
    "k_ladder":         K_LADDER,
}

# The one place this notebook's benchmark identity is chosen — every voting/
# grouping call downstream (generation-pass correctness, the Vote Once cell,
# the SC@k loop) takes SPEC and needs no other benchmark-specific branching.
from utils.voting import MMLU_SPEC as SPEC

W&B Project: agentic-final-results


## 4. Data Loading
We load the MMLU (`cais/mmlu`, `all` config) dataset. The fixed 128-question evaluation subset is drawn later (in section 10) with `random.seed(789)` — identical to the `v2.3.0` baseline's subset, so per-question results pair exactly. `problem_id` is simply each question's 0-based position in that seeded subset (the same identity scheme MATH-500 uses, no per-dataset branching).

In [5]:
from utils.data import load_mmlu

# ── MMLU DATASET ──────────────────────────────────────────────────────
mmlu_dataset = load_mmlu(split="test")

print(f"✅ MMLU dataset loaded with {len(mmlu_dataset)} examples.")
print(f"Example question: {mmlu_dataset[0]['question']}")
print(f"Example choices:  {mmlu_dataset[0]['choices']}")
print(f"Example answer:   {mmlu_dataset[0]['answer']}  (subject: {mmlu_dataset[0]['subject']})")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

✅ MMLU dataset loaded with 14042 examples.
Example question: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
Example choices:  ['0', '4', '2', '6']
Example answer:   1  (subject: abstract_algebra)


## 5. Model Loading (4-bit Quantized)

`load_model_and_tokenizer` loads Phi-4-mini-instruct via `AutoModelForCausalLM` and quantizes it to **NF4 4-bit** on the fly via bitsandbytes (~2.7 GB on a T4) — identical in shape to the `v2.3.0` baseline's call. Only this one model is ever resident: SC's generation pass is homogeneous (one live agent, 5 seeds), so `batch_size=1` and the single-model VRAM budget hold throughout.

In [6]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = CONFIG["model_id"]
model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)

print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

config.json: 0.00B [00:00, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

✅ Model loaded on cuda:0 — dtype torch.float16


## 6. Defining the Generation Chain

A single LangChain chain — the MMLU template (identical wording to `v2.3.0`, letter-choice answer format) piped to the LLM — is what each SC generator invokes. There is no Proposer/Critic/Refiner here either; the multi-agent structure in SC comes from *voting over 5 independent samples*, not from a multi-step pipeline.

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── MMLU Template (identical wording to the v2.3.0 baseline) ──────────
mmlu_template = PromptTemplate.from_template(
        "Your answer should be of the following format: 'ANSWER: [LETTER]' "
        "(without quotes) where [LETTER] is one of A,B,C,D. "
        "Think step by step before answering.\n\n"
        "{task}"
)

print("✅ MMLU chain template defined.")

✅ MMLU chain template defined.


## 7. Instrumented Generation Closure

`make_generate_fn` builds the `generate(question, seed) -> AgentSample` closure a **live** `AgentSpec` node needs (see `utils/graphs.py`): it times a single `chain.invoke`, counts input/output tokens with the model's own tokenizer (outside the timed region, exactly like the `v2.x` baselines' `run_agent_step`), and measures per-call peak VRAM. Section 10 builds one closure per generation seed and drives it through `utils.graphs.build_majority_voting_graph` — the same LangGraph template Majority Voting later reuses in replay mode.

In [8]:
import time
from utils.instrumentation import timed_invoke, tokens_per_second


def make_generate_fn(chain, prompt_template, model_key, model_id, agent_id, gen_id, extract_fn):
    """Build a live ``generate(task, seed) -> AgentSample`` closure for one
    SC generator pass.

    Instruments a single ``chain.invoke`` via ``timed_invoke`` (tokens counted
    outside the timed region, VRAM reset immediately before it) — the same
    sequence the v2.x baselines' ``run_agent_step`` hand-inlines. Returned dict
    matches ``utils.graphs.AgentSample``. ``extract_fn`` is ``SPEC.extract`` —
    this closure is otherwise benchmark-agnostic.
    """
    def generate(task, seed):
        r = timed_invoke(chain, prompt_template, tokenizer, {"task": task})

        print(
            f"  ⏱  gen_id={gen_id} seed={seed}: {r['latency_s']:.1f}s  |  "
            f"in={r['input_tokens']} out={r['output_tokens']} tok  |  peak={r['gpu_peak_mb']:.0f} MB"
        )

        return {
            "agent_id": agent_id, "model_key": model_key, "model_id": model_id, "source": "live",
            "gen_id": gen_id, "seed": seed,
            "answer": r["output_text"], "parsed_output": extract_fn(r["output_text"]),
            "latency_s": r["latency_s"], "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"], "total_tokens": r["total_tokens"],
            "gpu_peak_mb": r["gpu_peak_mb"],
        }
    return generate

print("✅ Generation closure builder ready.")

✅ Generation closure builder ready.


## 8. Model Warmup
Before timing, we "warm up" the model with 2 dummy executions so the one-time CUDA / bitsandbytes kernel compilation and cuBLAS autotuning happen here — not inside the first timed sample.

In [9]:
# ── Warmup ────────────────────────────────────────────────────────────
# Run a couple of throwaway generations through the same pipeline the
# evaluation uses, so one-time CUDA / bitsandbytes kernel compilation and
# cuBLAS autotuning happen here — not inside the first timed sample.
from utils.warmup import warmup_model

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=GEN["temperature"],
    top_p=GEN["top_p"], top_k=GEN["top_k"],
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'top_k', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔥 Warming up model (2 iterations, max_new_tokens=256)…


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Warmup complete in 9.7s — first evaluated sample will no longer pay cold-start cost.


9.733663976000003

## 9. Generation Run — Initialize Weights & Biases

We open a dedicated W&B run for the **generation pass** — the 5×128 live generations. This is separate from the four SC@k aggregation runs (section 13): keeping generation and aggregation as distinct runs lets W&B record real producer→consumer lineage (`use_artifact`), and lets the SC@k rule be re-run offline later with no GPU if the voting logic ever changes.

In [10]:
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics
from utils.registry import generation_run_name

run = init_wandb(
    run_name=generation_run_name(MODEL_KEY, DATASET),
    project=PROJECT,
    config={**CONFIG, **GEN, "eval_dataset": DATASET},
    tags=["l1", "self-consistency", "generation", DATASET],
)
print(f"✅ W&B run: {run.name}  —  {run.url}")

define_wandb_chart_only_metrics(prefixes=["mmlu_generation/*"])

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andreu-fernandez-cubi (tfg-optimizing-agentic-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260729_110501-o26qy5cb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run l1-sc-gen-SC-Phi-4-mini-instruct-mmlu
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/o26qy5cb


✅ W&B run: l1-sc-gen-SC-Phi-4-mini-instruct-mmlu  —  https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/o26qy5cb


## 10. Generation Pass

For each of the 5 seeds, we rebuild the LLM and chain with that seed, build a single-live-agent LangGraph (`build_majority_voting_graph([agent_spec], SPEC, rng=...)`), and invoke it once per question over the fixed 128-question subset. Every full response, its extracted letter, correctness (via `SPEC.score`/`SPEC.get_truth`), and instrumentation is appended to `generation_records` and streamed to W&B under `mmlu_sample/*`. After each completed seed pass, the accumulated records are re-logged as a `sc-generations-*` artifact **version** — a killed Kaggle session can resume from the last logged version instead of losing prior seeds' work.

In [11]:
import warnings
warnings.filterwarnings("ignore")  # ignore transformer warnings that clutter the eval output

import random
from tqdm.notebook import tqdm
from utils.agents import build_hf_langchain_llm
from utils.graphs import AgentSpec, build_majority_voting_graph
from utils.persistence import make_generation_record, log_generations_artifact, artifact_ref
from utils.registry import generations_artifact_name
from utils.logging_utils import log_metrics

sample_size = wandb.config.eval_size

# Fix the evaluation subset once — identical to the v2.3.0 baseline's own subset
# (same data_seed, same eval_size) -> the SC pool pairs exactly with it.
random.seed(wandb.config.data_seed)
test_sample = random.sample(list(mmlu_dataset), sample_size)

# A single agent never ties (nothing to vote against), so this RNG never
# actually draws — it exists only to satisfy build_majority_voting_graph's
# signature. The Vote Once cell gets its own fresh, independently seeded RNG.
gen_tie_rng = random.Random(TIE_BREAK_SEED)

generation_records = []
producer_ref = None

for gen_id, seed in enumerate(SC_SEEDS):
    print(f"\n{'='*60}")
    print(f"  Generator {gen_id + 1}/{len(SC_SEEDS)}  —  seed={seed}")
    print(f"{'='*60}")

    llm = build_hf_langchain_llm(
        model=model, tokenizer=tokenizer,
        max_new_tokens=GEN["max_new_tokens"], temperature=GEN["temperature"],
        top_p=GEN["top_p"], top_k=GEN["top_k"],
        return_full_text=False, seed=seed,
    )
    chain = mmlu_template | llm | StrOutputParser()

    generate_fn = make_generate_fn(chain, mmlu_template, MODEL_KEY, CONFIG["model_id"], "gen", gen_id, SPEC.extract)
    agent_spec = AgentSpec(agent_id="gen", model_key=MODEL_KEY, model_id=CONFIG["model_id"], mode="live", generate=generate_fn, seed=seed)
    graph = build_majority_voting_graph([agent_spec], SPEC, rng=gen_tie_rng)

    for problem_id, item in enumerate(tqdm(test_sample, desc=f"{DATASET} gen_id={gen_id} seed={seed} ({sample_size} samples)")):
        task         = SPEC.render_task(item)
        ground_truth = SPEC.get_truth(item)

        out = graph.invoke({
            "task": task, "problem_id": problem_id,
            "dataset": DATASET, "samples": [], "vote": None, "metrics": None,
        })
        sample = out["samples"][0]
        compare_value = sample["answer"] if SPEC.compare_on == "answer" else sample["parsed_output"]
        correct = SPEC.score(compare_value, ground_truth)
        tps     = tokens_per_second(sample["output_tokens"], sample["latency_s"])

        generation_records.append(make_generation_record(
            dataset=DATASET, problem_id=problem_id, model_key=MODEL_KEY, gen_id=gen_id,
            seed=seed, protocol="generation", task=task, sample=sample,
            correct=correct, tokens_per_s=tps,
        ))

        log_metrics({
            "mmlu_generation/accuracy":       int(correct),
            "mmlu_generation/latency_s":      round(sample["latency_s"], 2),
            "mmlu_generation/output_tokens":  sample["output_tokens"],
            "mmlu_generation/total_tokens":   sample["total_tokens"],
            "mmlu_generation/gpu_peak_mb":    round(sample["gpu_peak_mb"], 2),
            "mmlu_generation/throughput_tps": round(tps, 2),
        })

    # Re-log the artifact after every completed seed pass (crash-resumable sessions).
    manifest = {
        "model_key": MODEL_KEY, "dataset": DATASET,
        "protocol": "self-consistency", "data_seed": wandb.config.data_seed,
        "eval_size": sample_size, "generation_seeds": SC_SEEDS[:gen_id + 1],
        "gen_params": GEN, "wandb_run_id": run.id, "wandb_run_name": run.name,
        "wandb_project": PROJECT, "n_records": len(generation_records),
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    artifact = log_generations_artifact(
        generation_records, manifest,
        name=generations_artifact_name(MODEL_KEY, DATASET), run=run,
    )
    # Captured off the actually-committed version (artifact.wait()'d inside
    # log_generations_artifact), not computed as ":v{n_generators-1}" — a
    # resumed or partial session would otherwise silently repoint this at the
    # wrong version. Recorded into run.summary immediately, so it survives a
    # kill between seed passes.
    producer_ref = artifact_ref(artifact, entity=run.entity, project=PROJECT)
    wandb.summary["producer_artifact"] = producer_ref
    wandb.summary["producer_artifact_version"] = artifact.version
    print(f"✅ Generations artifact re-logged after seed pass {gen_id + 1}/{len(SC_SEEDS)} -> {producer_ref}")

# No display table here: the sc-generations-* artifact logged above is the one
# place full generations + metrics live, and MAJ/BoN-MAV already read it via
# that artifact — a Table here would just duplicate it.
from utils.logging_utils import finish_wandb
finish_wandb()

print(f"\n✅ Generation pass complete: {len(generation_records)} records "
      f"({sample_size} problems x {len(SC_SEEDS)} generators).")


  Generator 1/5  —  seed=42


mmlu gen_id=0 seed=42 (128 samples):   0%|          | 0/128 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  gen_id=0 seed=42: 5.8s  |  in=81 out=92 tok  |  peak=2972 MB


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  ⏱  gen_id=4 seed=987: 7.7s  |  in=102 out=131 tok  |  peak=2976 MB
  ⏱  gen_id=4 seed=987: 5.9s  |  in=86 out=98 tok  |  peak=2973 MB


wandb: updating run metadata


✅ Generations artifact re-logged after seed pass 5/5 -> tfg-optimizing-agentic-ai/agentic-final-results/sc-generations-phi-4-mini-mmlu:v4


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:       mmlu_generation/accuracy ▁█▁██▁█▁▁▁██▁███▁██▁████▁▁█▁████▁███▁█▁▁
wandb:    mmlu_generation/gpu_peak_mb ▅▁▁▁▁▅▂█▅▁▅█▁▂▁▁▁▁▅▅▇██▇▁▁▇▅▅▇▅▁▁▅▁█▁▂▁▂
wandb:      mmlu_generation/latency_s ▂▆▁▃▃▆▁▄▄▁▄▂▂▂█▂▄▂▃▃▃▅▇▃▃▄▄▃▃▂▁▁▂▂▃▃▅▃▃▃
wandb:  mmlu_generation/output_tokens ▁▁▅▅▃▃▂▃▄▅▁▆▄▃▁▁▃▃▅▃▃▅▅▃▅▄▃▅▄▄▄▄▃█▄▄▆▃▃▄
wandb: mmlu_generation/throughput_tps ▃▁██▇██▇███▇▄█▇████▄▇▇████▇█▇██▇▇▇▇███▄▅
wandb:   mmlu_generation/total_tokens ▆▃▃▂▂▃██▇▁▂▅▃▅▂▃▃▃▃▃▂▂▇▃▆▃▃▆▁▁▂▂▂▂▄▃▄▂▁▂
wandb: 
wandb: Run summary:
wandb:         producer_artifact tfg-optimizing-agent...
wandb: producer_artifact_version v4
wandb: 
wandb: 🚀 View run l1-sc-gen-SC-Phi-4-mini-instruct-mmlu at: https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/o26qy5cb
wandb: ⭐️ View project at: https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results
wandb: Synced 5 W&B file(s), 0 media file(s), 15 artifact


✅ Generation pass complete: 640 records (128 problems x 5 generators).


## 11. Run the Self-Consistency Ladder

`utils.voting.run_self_consistency(store, test_sample, SPEC, K_LADDER, ...)` drives the whole SC@1/3/4/5 ladder in one call: it compiles one replay LangGraph per subset of the ladder (21 graphs for 5 generators), scores every generation against ground truth once via `SPEC.score`, then invokes each subset's own graph — which groups (exact letter match, case/whitespace-insensitive) and votes over just that subset's members. Every vote draws from one fresh `random.Random(TIE_BREAK_SEED)`, so this cell alone is independently reproducible without re-running the (expensive, GPU-bound) generation pass above it — the documented "re-run SC@k offline, no GPU" path.

In [12]:
from utils.persistence import build_generation_store
from utils.voting import run_self_consistency

store        = build_generation_store(generation_records)
n_generators = len(SC_SEEDS)

# Fresh, independently seeded RNG for this cell — reproducible on its own,
# whether or not the generation pass above it was re-run this session.
vote_tie_rng = random.Random(TIE_BREAK_SEED)

sc_results = run_self_consistency(
    store, test_sample, SPEC, K_LADDER,
    n_generators=n_generators, rng=vote_tie_rng,
    model_key=MODEL_KEY, model_id=CONFIG["model_id"],
)

print(f"✅ Ran the SC@{{{','.join(str(k) for k in K_LADDER)}}} ladder over "
      f"{sample_size} problems x {n_generators} generations.")

✅ Ran the SC@{1,3,4,5} ladder over 128 problems x 5 generations.


## 12. SC@k — Aggregation Runs

One W&B run per `k ∈ {1, 3, 4, 5}`. Each run consumes the generation artifact via `use_artifact` (real lineage), reads that k's already-computed subset results off `sc_results` (from the previous cell — the graph protocol produced every vote), and composes each subset's metrics with `utils.voting.compose_generation_metrics` (latency/tokens summed across the subset's generators, VRAM = max — the single-model case of the compose function MAJ/BoN-MAV also share). Per-question, per-subset results become a `(sample_size, C(5,k))` matrix — the exact `utils.statistics` estimators unchanged from the `v2.x` baselines apply directly, since they already reduce to per-question means. Each run gets its own results table (`results_sc{k}_MMLU`, one row per SC result, no `output` column, adds `protocol`/`pct_votes_top`) and `evaluation-matrices-*-SC{k}-MMLU` artifact, mirroring the 2.0.0 convention.

In [13]:
import math
import tempfile
import numpy as np
from utils.voting import compose_generation_metrics
from utils.registry import matrices_artifact_name, sc_run_name, problem_ref
from utils.persistence import load_generations_artifact, build_generation_store
from utils.logging_utils import log_stat_summary, finish_wandb
from utils.statistics import (
    accuracy_statistics, time_statistics, token_statistics, vram_statistics,
    print_accuracy_stats_report, print_time_report, print_token_report, print_vram_report,
)

# producer_ref is the actually-committed version captured at the end of
# section 10 (utils.persistence.artifact_ref, off the .wait()'d artifact) —
# not a computed ":v{n-1}" guess, which would silently repoint at the wrong
# version after any resumed/partial generation run. Each sc_run re-downloads
# it via load_generations_artifact(run=sc_run) rather than a bare
# use_artifact call, so every one of the 4 aggregation runs records its own
# real producer->consumer lineage (Claim-to-Evidence), not just the
# generation run's in-memory records.

for k in K_LADDER:
    n_subsets = math.comb(n_generators, k)
    print(f"\n{'='*60}")
    print(f"  SC@{k}  —  {n_subsets} subset(s) per problem")
    print(f"{'='*60}")

    sc_run = init_wandb(
        run_name=sc_run_name(MODEL_KEY, DATASET, k),
        project=PROJECT,
        config={**CONFIG, **GEN, "eval_dataset": DATASET, "sc_k": k, "n_subsets": n_subsets,
                "producer_artifact": producer_ref},
        tags=["l1", "self-consistency", f"sc{k}", DATASET],
    )
    records, manifest = load_generations_artifact(producer_ref, run=sc_run)
    store = build_generation_store(records)
    define_wandb_chart_only_metrics(prefixes=["mmlu_sample/*"])

    correctness_matrix = np.zeros((sample_size, n_subsets), dtype=int)
    time_matrix         = np.zeros((sample_size, n_subsets), dtype=float)
    input_tok_matrix    = np.zeros((sample_size, n_subsets), dtype=int)
    output_tok_matrix   = np.zeros((sample_size, n_subsets), dtype=int)
    total_tok_matrix    = np.zeros((sample_size, n_subsets), dtype=int)
    gpu_peak_matrix      = np.zeros((sample_size, n_subsets), dtype=float)
    tok_per_s_matrix     = np.zeros((sample_size, n_subsets), dtype=float)

    results_columns = [
        "dataset", "model_key", "protocol", "problem_id", "subset_id",
        "problem_ref", "winner_gen_id", "seeds",
        "task", "ground_truth", "final_output", "correct",
        "pct_votes_top", "tie", "n_abstain",
        "latency_s", "output_tokens", "total_tokens", "gpu_peak_mb", "throughput_tok_s",
    ]
    results_rows = []
    n_tie = 0
    n_abstain_total = 0

    for problem_id, item in enumerate(test_sample):
        subset_results = sc_results[problem_id]["by_k"][k]
        records        = [store[(problem_id, gen_id)] for gen_id in range(n_generators)]
        ground_truth   = SPEC.get_truth(item)
        task           = SPEC.render_task(item)
        ref            = problem_ref(DATASET, problem_id)

        for subset_idx, subset_result in enumerate(subset_results):
            gen_metrics = [records[i] for i in subset_result["indices"]]
            metrics = compose_generation_metrics(gen_metrics)

            correctness_matrix[problem_id, subset_idx] = int(subset_result["correct"])
            time_matrix[problem_id, subset_idx]        = metrics["latency_s"]
            input_tok_matrix[problem_id, subset_idx]   = metrics["input_tokens"]
            output_tok_matrix[problem_id, subset_idx]  = metrics["output_tokens"]
            total_tok_matrix[problem_id, subset_idx]   = metrics["total_tokens"]
            gpu_peak_matrix[problem_id, subset_idx]     = metrics["gpu_peak_mb"]
            tok_per_s_matrix[problem_id, subset_idx]    = metrics["tokens_per_s"]

            if subset_result["tie"]:
                n_tie += 1
            n_abstain_total += subset_result["n_abstain"]

            seeds_str = ",".join(str(SC_SEEDS[i]) for i in subset_result["indices"])

            results_rows.append([
                DATASET, MODEL_KEY, f"SC@{k}", problem_id, subset_result["subset_id"],
                ref, subset_result["winner_gen_id"], seeds_str,
                task, ground_truth, subset_result["winner_output"], subset_result["correct"],
                round(subset_result["pct_votes_top"], 4), subset_result["tie"], subset_result["n_abstain"],
                round(metrics["latency_s"], 2), metrics["output_tokens"], metrics["total_tokens"],
                round(metrics["gpu_peak_mb"], 2), round(metrics["tokens_per_s"], 4),
            ])

        # One chart point per problem — the mean across that problem's
        # C(n_generators, k) subsets, not one point per subset — so the
        # chart stays at `sample_size` points per run regardless of k.
        log_metrics({
            "mmlu_sample/accuracy":         round(float(correctness_matrix[problem_id].mean()), 4),
            "mmlu_sample/latency_s":        round(float(time_matrix[problem_id].mean()), 2),
            "mmlu_sample/output_tokens":    round(float(output_tok_matrix[problem_id].mean()), 1),
            "mmlu_sample/total_tokens":     round(float(total_tok_matrix[problem_id].mean()), 1),
            "mmlu_sample/gpu_peak_mb":      round(float(gpu_peak_matrix[problem_id].mean()), 2),
            "mmlu_sample/throughput_tok_s": round(float(tok_per_s_matrix[problem_id].mean()), 4),
        })

    # results_rows are already emitted in (problem_id, subset_idx) order by
    # the loop above — no re-sort needed (a sort keyed on positional column
    # indices is exactly what breaks silently the next time columns move).
    results_table = wandb.Table(columns=results_columns, data=results_rows)
    wandb.log({f"results_sc{k}_MMLU": results_table})

    # ── Statistics (same estimators, same summary keys as the 2.x baselines) ──
    acc_stats   = accuracy_statistics(correctness_matrix)
    time_stats  = time_statistics(time_matrix)
    out_stats   = token_statistics(output_tok_matrix, max_tokens=GEN["max_new_tokens"])
    total_stats = token_statistics(total_tok_matrix)
    vram_stats  = vram_statistics(gpu_peak_matrix)

    print_accuracy_stats_report(f"SC@{k} Accuracy (MMLU)", acc_stats)
    print_time_report(f"SC@{k} Latency (MMLU)", time_stats)
    print_token_report(f"SC@{k} Output Tokens (MMLU)", out_stats)
    print_token_report(f"SC@{k} Total Tokens (MMLU)", total_stats)
    print_vram_report(f"SC@{k} Peak VRAM (MMLU)", vram_stats)

    wandb.summary["n_generators"]      = n_generators
    wandb.summary["n_subsets"]         = n_subsets
    wandb.summary["n_problems"]        = sample_size
    wandb.summary["total_experiments"] = sample_size * n_subsets
    wandb.summary["accuracy"]          = round(acc_stats["p_hat"], 6)
    wandb.summary["tie_rate"]          = round(n_tie / (sample_size * n_subsets), 4)
    wandb.summary["abstain_rate"]      = round(n_abstain_total / (sample_size * n_subsets * n_generators), 4)

    log_stat_summary("accuracy_stats", acc_stats,
        keys=["ci_lower", "ci_upper", "between_problem_std"])
    wandb.summary["latency"] = round(time_stats["mean"], 2)
    log_stat_summary("latency_stats", time_stats, keys=["ci_lower", "ci_upper", "between_problem_std", "flat_cv"])
    wandb.summary["output_tokens"] = round(out_stats["mean"], 1)
    log_stat_summary("output_tokens_stats", out_stats,
        keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index", "frac_at_ci_lower", "frac_at_ci_upper"])
    wandb.summary["total_tokens"] = round(total_stats["mean"], 1)
    log_stat_summary("total_tokens_stats", total_stats, keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])
    wandb.summary["mean_vram_peak"] = round(vram_stats["mean_peak"], 0)
    wandb.summary["max_vram_peak"]  = round(vram_stats["peak_max"], 0)
    log_stat_summary("vram_peak_stats", vram_stats, keys=["peak_ci_lower", "peak_ci_upper", "between_problem_peak_std"])

    # ── Evaluation-matrices artifact (2.x convention, sc{k}_-prefixed keys) ──
    artifact = wandb.Artifact(
        matrices_artifact_name(MODEL_KEY, f"sc{k}", DATASET),
        type="evaluation-data",
        description=f"SC@{k} evaluation matrices ({sample_size} x {n_subsets}). MMLU with {MODEL_KEY}.",
    )
    with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
        np.savez(
            f.name,
            **{
                "problem_ids":          np.arange(sample_size),
                f"sc{k}_correctness":   correctness_matrix,
                f"sc{k}_times":         time_matrix,
                f"sc{k}_input_tokens":  input_tok_matrix,
                f"sc{k}_output_tokens": output_tok_matrix,
                f"sc{k}_total_tokens":  total_tok_matrix,
                f"sc{k}_gpu_peak":      gpu_peak_matrix,
            },
        )
        artifact.add_file(f.name, name="evaluation_matrices.npz")
    wandb.log_artifact(artifact)

    finish_wandb()
    print(f"✅ SC@{k} run finished.")

print("\n✅ Full SC@1/3/4/5 ladder complete.")


  SC@1  —  5 subset(s) per problem


wandb: setting up run 6p7ecknk
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260729_122628-6p7ecknk
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run l1-sc@1-SC-Phi-4-mini-instruct-mmlu
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/6p7ecknk
wandb:   2 of 2 files downloaded.  



  SC@1 Accuracy (MMLU)
  Accuracy p̂                  60.00%
  95% Cluster-t CI         [52.38, 67.62]%
  Between-problem std        43.5438%
  ── Supplementary: Wilson ──
  95% Wilson CI              [52.31, 67.23]%
  DEFF                       162.020
  ICC                          0.738
  ── Sample ──
  Problems x Seeds           128 x 5  (n=640)


  SC@1 Latency (MMLU)
  Mean (s)                      7.5950
  Std (s)                       5.7298
  CV                            0.7544
  95% BCa CI (s)             [6.9092, 8.5238]  [cluster_bca]
  Between-problem std (s)       4.6629
  ── Percentiles (descriptive) ──
  Median / P50 (s)              6.5976
  P25 (s)                       4.6319
  P75 (s)                       9.2578
  IQR (s)                       4.6259
  P95 (s)                      18.1719
  ── Sample ──
  Problems x Seeds           128 x 5  (n=640)


  SC@1 Output Tokens (MMLU)
  Mean tok                        125.7031
  Std tok                          97.1489


wandb: updating run metadata; uploading artifact run-6p7ecknk-results_sc1_MMLU; uploading artifact evaluation-matrices-phi-4-mini-sc1-mmlu
wandb: uploading artifact run-6p7ecknk-results_sc1_MMLU; uploading artifact evaluation-matrices-phi-4-mini-sc1-mmlu
wandb: uploading artifact evaluation-matrices-phi-4-mini-sc1-mmlu
wandb: uploading history steps 0-128, summary, console lines 0-92
wandb: 
wandb: Run history:
wandb:         mmlu_sample/accuracy ▁▄█▁▁████▁▁█▄▁█▂▁█▅█▂▇▁█▂█▁█▇█▁█▁███▅██▅
wandb:      mmlu_sample/gpu_peak_mb ▅▁▅▁▁▅▁▁▆▁▁▁▂█▁▇▁▆▅▂▁▆▁▁▁▇█▅▂▁▅▅▁▁▅▁▁▁▁▁
wandb:        mmlu_sample/latency_s ▂▂▂▂▂▄▄▂▁▃▂▅▁▄▁▂▂▂▁▁▂▂▃▃▂▄▂▂▁█▂▁▂▃▃▂▂▂▃▂
wandb:    mmlu_sample/output_tokens ▂▇▂▂▄▇▄▇▂▃▂▂▆▁▂▂▂▁▁▃▂▇▃▃▃▂█▃▂▄▅▃▂▂▂▂▂▅▁▄
wandb: mmlu_sample/throughput_tok_s ▄▁▇▅▇▇▇▇▇▇▆▇▃▇▇▂▅▆▇▇▇▇▅▂▇▇▆▇▇▂▆▆▆█▇▇▇█▇▇
wandb:     mmlu_sample/total_tokens ▂▂▄▂▅▂▆▄▃▄▄▂▂▄▁▂▂▂▂▂▄▅▃▃▂▂▃▅█▂▅▂▃▃▃▂▃▂▃▃
wandb: 
wandb: Run summary:
wandb:                       abstain_rate 0.0003
wandb:                           accuracy 0.6


✅ SC@1 run finished.

  SC@3  —  10 subset(s) per problem


...

✅ SC@4 run finished.

  SC@5  —  1 subset(s) per problem


wandb: setting up run sjjsa1pm
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260729_122646-sjjsa1pm
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run l1-sc@5-SC-Phi-4-mini-instruct-mmlu
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-ai/agentic-final-results/runs/sjjsa1pm
wandb:   2 of 2 files downloaded.  



  SC@5 Accuracy (MMLU)
  Accuracy p̂                  60.94%
  95% Cluster-t CI         [52.37, 69.50]%
  Between-problem std        48.9808%
  ── Supplementary: Wilson ──
  95% Wilson CI              [52.25, 68.98]%
  DEFF                       127.000
  ICC                          0.000
  ── Sample ──
  Problems x Seeds           128 x 1  (n=128)


  SC@5 Latency (MMLU)
  Mean (s)                     37.9752
  Std (s)                      23.3144
  CV                            0.6139
  95% BCa CI (s)             [34.5458, 42.6191]  [cluster_bca]
  Between-problem std (s)      23.3144
  ── Percentiles (descriptive) ──
  Median / P50 (s)             32.7695
  P25 (s)                      25.0136
  P75 (s)                      43.2707
  IQR (s)                      18.2571
  P95 (s)                      82.8111
  ── Sample ──
  Problems x Seeds           128 x 1  (n=128)


  SC@5 Output Tokens (MMLU)
  Mean tok                        628.5156
  Std tok                         395.282

wandb: updating run metadata; uploading console lines 0-0; uploading artifact run-sjjsa1pm-results_sc5_MMLU; uploading artifact evaluation-matrices-phi-4-mini-sc5-mmlu
wandb: uploading artifact run-sjjsa1pm-results_sc5_MMLU; uploading artifact evaluation-matrices-phi-4-mini-sc5-mmlu
wandb: uploading artifact run-sjjsa1pm-results_sc5_MMLU
wandb: uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading media/table/results_sc5_MMLU_128_6e19356e91c98bbdf034.table.json (+ 1 more)
wandb: 
wandb: Run history:
wandb:         mmlu_sample/accuracy █▁▁▁███▁██▁██▁███████▁▁█████▁▁████████▁▁
wandb:      mmlu_sample/gpu_peak_mb ▅▂▅█▁▁▄▁█▁▁▂▁▁▅▁▁▅▁▂▁▂▁▅▁▁▅▆▅▅▆▁█▅▂▁▁▁▁▁
wandb:        mmlu_sample/latency_s ▂▂▁▃▃▇▃▃▄▃█▄▃▂▃▂▂▃▁▁▂▇▄▅▄▃▄▆▃▃▃▂▄▂▁▁▃▂▂▂
wandb:    mmlu_sample/output_tokens ▃▃▂▂▃▃▄▁▅▃▄▅▃▄▃▄▁▃▃▆▃▃▄▂▂▂▄▃▂▃▄█▃▂▅▂▂▃▃▂
wandb: mmlu_sample/throughput_tok_s █▇▇▇█▇▁████▇██▆▂███▇▇▄██▇▇███▇▇████▇▇▇█▇
wandb:     mmlu_sample/total_tokens ▁▃▂▁▂▄

✅ SC@5 run finished.

✅ Full SC@1/3/4/5 ladder complete.


## 13. Analysis & Next Steps

### What W&B gives you (5 runs from this notebook)
- **1 generation run** (`l1-sc-gen-*`) — the live 5×128 generations, per-generator instrumentation, and the untruncated `sc-generations-*` artifact (the sole reuse vehicle for the full generations + their metrics — no separate display table, so nothing duplicates it).
- **4 aggregation runs** (`l1-sc1/3/4/5-*`) — the SC@1/3/4/5 ladder, each with 2.0.0-parity per-question graphs, cluster-robust accuracy/latency/token/VRAM CIs, a `results_sc{k}_MMLU` table, and an `evaluation-matrices-*-SC{k}-MMLU` artifact. Each records `use_artifact` lineage back to the generation run.
- **Diagnostics**: `tie_rate` and `abstain_rate` in every aggregation run's summary — a flat SC@k ladder or a high tie rate is directly visible there rather than being mysterious.

### Expected results
SC@1 (mean of the 5 individual generations) should sit close to — but not identical to — the `v2.3.0` baseline's MMLU accuracy: same model, data subset and most sampling params, but `top_k=40` instead of `20`. MMLU's single-letter answer space is smaller than MATH-500's, so expect a **higher tie rate and less headroom for SC@k to improve over SC@1** — watch `pct_votes_top` for signs of degenerate (all-same-letter) voting.

### Role of this notebook
This is the **only fresh generator compute** in the L1 campaign for Phi-4-mini-instruct on MMLU. Majority Voting and BoN-MAV notebooks consume `sc-generations-phi-4-mini-mmlu` directly — no regeneration.

### Next steps
1. Once all 4 models × 2 benchmarks have run, build the Majority Voting notebooks (`notebooks/majority-voting/`, v4.x) — pure replay, no GPU.